In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../data/Airline_review.csv")

In [4]:
df = df.dropna(subset=["Overall_Rating"])

In [6]:
df.shape

(23171, 20)

In [7]:
df["Recommended"].unique()

<StringArray>
['yes', 'no']
Length: 2, dtype: str

In [8]:
df["full_review_text"] = df["Review_Title"] + " " + df["Review"]
df[["Review_Title", "Review", "full_review_text"]].head()

,Review_Title,Review,full_review_text
0,"""pretty decent airline""",Moroni to Moheli. Turned out to be a pretty ...,"""pretty decent airline"" Moroni to Moheli. Tu..."
1,"""Not a good airline""",Moroni to Anjouan. It is a very small airline...,"""Not a good airline"" Moroni to Anjouan. It is..."
2,"""flight was fortunately short""",Anjouan to Dzaoudzi. A very small airline an...,"""flight was fortunately short"" Anjouan to Dz..."
3,"""I will never fly again with Adria""",Please do a favor yourself and do not fly wi...,"""I will never fly again with Adria"" Please d..."
4,"""it ruined our last days of holidays""",Do not book a flight with this airline! My fr...,"""it ruined our last days of holidays"" Do not ..."


In [9]:
import re
def clean_text(text):
    
    text = str(text)
    
    # lowercase
    text = text.lower()
    
    # remove urls
    text = re.sub(r"http\S+", "", text)
    
    # remove special characters
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    
    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [13]:
df["clean_review_text"] = df["full_review_text"].apply(clean_text)

In [10]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [15]:
def remove_stopwords(text):

    words = text.split()

    filtered_words = [word for word in words if word not in stop_words]

    return " ".join(filtered_words)

In [16]:
df["processed_text"] = df["clean_review_text"].apply(remove_stopwords)
df[["clean_review_text","processed_text"]].head()


,clean_review_text,processed_text
0,pretty decent airline moroni to moheli turned ...,pretty decent airline moroni moheli turned pre...
1,not a good airline moroni to anjouan it is a v...,good airline moroni anjouan small airline tick...
2,flight was fortunately short anjouan to dzaoud...,flight fortunately short anjouan dzaoudzi smal...
3,i will never fly again with adria please do a ...,never fly adria please favor fly adria route m...
4,it ruined our last days of holidays do not boo...,ruined last days holidays book flight airline ...


In [17]:
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [18]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):

    words = text.split()

    lemmatized = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(lemmatized)

In [19]:
df["final_text"] = df["processed_text"].apply(lemmatize_text)

In [20]:
df[["processed_text", "final_text"]].head()

,processed_text,final_text
0,pretty decent airline moroni moheli turned pre...,pretty decent airline moroni moheli turned pre...
1,good airline moroni anjouan small airline tick...,good airline moroni anjouan small airline tick...
2,flight fortunately short anjouan dzaoudzi smal...,flight fortunately short anjouan dzaoudzi smal...
3,never fly adria please favor fly adria route m...,never fly adria please favor fly adria route m...
4,ruined last days holidays book flight airline ...,ruined last day holiday book flight airline fr...


In [26]:
df.to_csv("../data/clean_airline_reviews.csv", index=False)

In [27]:
df["Overall_Rating"] = pd.to_numeric(df["Overall_Rating"], errors="coerce")

In [22]:
def severity_label(rating):

    if rating <= 2:
        return "Critical"
    
    elif rating <= 4:
        return "High"
    
    elif rating <= 6:
        return "Medium"
    
    else:
        return "Low"

In [28]:
df["severity_label"] = df["Overall_Rating"].apply(severity_label)

In [29]:
df["severity_label"].value_counts()

severity_label
Critical    13891
Low          5560
High         2215
Medium       1505
Name: count, dtype: int64

In [30]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["severity_id"] = label_encoder.fit_transform(df["severity_label"])

In [31]:
df[["severity_label","severity_id"]].head()

,severity_label,severity_id
0,Low,2
1,Critical,0
2,Critical,0
3,Critical,0
4,Critical,0
